In [62]:
from langgraph.graph import StateGraph,START,END
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from typing import TypedDict,Annotated
from pydantic import BaseModel, Field
import operator

In [63]:
load_dotenv()

True

In [64]:
model = ChatGoogleGenerativeAI(model='gemini-2.5-flash-lite')

In [65]:
class EvaluateSchema(BaseModel):

    feedback : str = Field(description='Detailed feedback for the essay')
    score : int = Field(description='Score out of 10', ge=0,le=10)

In [66]:
Structure_model = model.with_structured_output(EvaluateSchema)

In [67]:
essay = """India in 2026 is experiencing intensifying heat as climate change, rapid urbanization, and environmental degradation combine to make extreme temperatures more frequent and severe. Once known for varied seasonal patterns, many regions now face prolonged heatwaves, rising average temperatures, and record-breaking summers. Cities such as Delhi, Jaipur, Nagpur, and Hyderabad often endure dangerous heat for extended periods, with temperatures crossing 45°C more regularly.

One major reason is global warming, driven by greenhouse gas emissions. Rising global temperatures have increased the intensity of heatwaves across South Asia. In India, deforestation, shrinking green cover, and expanding concrete urban areas worsen the problem through the urban heat island effect, where cities trap more heat than surrounding rural areas.

This shift has serious consequences. Agriculture is under pressure as crop yields decline due to water scarcity and heat stress. Public health risks are growing, especially for outdoor workers, children, and the elderly, with cases of heatstroke and dehydration increasing. Energy demand for cooling systems has surged, placing additional strain on electricity infrastructure.

Despite these challenges, India is also responding. Government initiatives include heat action plans, renewable energy expansion, improved weather forecasting, and urban greening projects. Public awareness about hydration, sustainable living, and environmental conservation is increasing.

India’s transformation into a hotter country in 2026 highlights the urgent need for climate resilience, sustainable development, and collective action. The country’s future will depend on balancing economic growth with environmental responsibility to protect both people and ecosystems from escalating heat."""

In [68]:
prompt = f"Evaluete the language quality of the following eassya and provide a feedback and assign a score out of 10 \n{essay}"
print(Structure_model.invoke(prompt).feedback)
print(Structure_model.invoke(prompt).score)

The essay effectively describes the multifaceted issue of rising temperatures in India by 2026, linking climate change, urbanization, and environmental degradation. The language is clear and concise, with a logical flow of ideas from cause to consequence to response. The essay identifies key cities affected and outlines specific impacts on agriculture, public health, and energy. The mention of government initiatives and public awareness adds a balanced perspective. The conclusion effectively summarizes the challenges and calls for action. Overall, the language quality is good, with strong topic sentences and supporting details.
9


In [69]:
class UPSCState(TypedDict):

    essay: str
    language_feedback : str
    analysis_feedback : str
    clarity_feedback : str
    overall_feedback : str
    individual_score : Annotated[list[int],operator.add]
    avg_score : float

In [70]:
def language_feedback(state:UPSCState):
    prompt = f"Evaluete the language quality of the following eassya and provide a feedback and assign a score out of 10 \n{essay}"
    output = Structure_model.invoke(prompt)

    return {'language_feedback':output.feedback, 'individual_score':[output.score]}

In [71]:
def depth_of_analysis(state:UPSCState):
    prompt = f"Evaluete the depth of analysisof the following eassya and provide a feedback and assign a score out of 10 \n{essay}"
    output = Structure_model.invoke(prompt)

    return {'analysis_feedback':output.feedback, 'individual_score':[output.score]}

In [72]:
def clearity_of_thought(state:UPSCState):
    prompt = f"Evaluete the clearity of thought of the following eassya and provide a feedback and assign a score out of 10 \n{essay}"
    output = Structure_model.invoke(prompt)

    return {'clarity_feedback':output.feedback, 'individual_score':[output.score]}

In [73]:
def final_evaluation(state: UPSCState):

    # summary feedback
    prompt = f'Based on the following feedbacks create a summarized feedback \n language feedback - {state["language_feedback"]} \n depth of analysis feedback - {state["analysis_feedback"]} \n clarity of thought feedback - {state["clarity_feedback"]}'
    overall_feedback = model.invoke(prompt).content

    # avg calculate
    avg_score = sum(state['individual_score'])/len(state['individual_score'])

    return {'overall_feedback': overall_feedback, 'avg_score': avg_score}

In [74]:

graph = StateGraph(UPSCState)

# Use consistent names in add_node and add_edge
graph.add_node('language_feedback', language_feedback)
graph.add_node('depth_of_analysis', depth_of_analysis)
graph.add_node('clearity_of_thought', clearity_of_thought)
graph.add_node('final_evaluation', final_evaluation)

# These names must exactly match the node names above
graph.add_edge(START, 'language_feedback')
graph.add_edge(START, 'depth_of_analysis')
graph.add_edge(START, 'clearity_of_thought')

graph.add_edge('language_feedback', 'final_evaluation')
graph.add_edge('depth_of_analysis', 'final_evaluation')
graph.add_edge('clearity_of_thought', 'final_evaluation')

graph.add_edge('final_evaluation', END)

workflow = graph.compile()



In [75]:
initial_state = {'essay':essay}
workflow.invoke(initial_state)



{'essay': 'India in 2026 is experiencing intensifying heat as climate change, rapid urbanization, and environmental degradation combine to make extreme temperatures more frequent and severe. Once known for varied seasonal patterns, many regions now face prolonged heatwaves, rising average temperatures, and record-breaking summers. Cities such as Delhi, Jaipur, Nagpur, and Hyderabad often endure dangerous heat for extended periods, with temperatures crossing 45°C more regularly.\n\nOne major reason is global warming, driven by greenhouse gas emissions. Rising global temperatures have increased the intensity of heatwaves across South Asia. In India, deforestation, shrinking green cover, and expanding concrete urban areas worsen the problem through the urban heat island effect, where cities trap more heat than surrounding rural areas.\n\nThis shift has serious consequences. Agriculture is under pressure as crop yields decline due to water scarcity and heat stress. Public health risks are 